# Part 2 — Unsupervised Learning

**Dataset**: AI4I 2020 Predictive Maintenance  
**Tasks**:
1. Dimensionality Reduction: PCA and t-SNE
2. Clustering: K-Means and DBSCAN
3. Hierarchical Clustering (Dendrogram)
4. Comparative Analysis


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
print('Libraries loaded.')

## 1. Data Loading and Preparation


In [ ]:
from src.data.loader import load_raw
from src.data.preprocessor import engineer_features

df = engineer_features(load_raw())

FEATURES = [
    'Air temperature [K]', 'Process temperature [K]',
    'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
    'temp_diff', 'power', 'wear_rate'
]

X = df[FEATURES].values
y_true = df['Machine failure'].values
machine_type = df['Type_encoded'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Feature matrix: {X_scaled.shape}')
print(f'Failure rate: {y_true.mean()*100:.2f}%')

## 2. PCA — Principal Component Analysis

Finds orthogonal directions of maximum variance.


In [ ]:
pca_full = PCA().fit(X_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
n_85 = int(np.argmax(cum_var >= 0.85)) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(pca_full.explained_variance_ratio_)+1),
            pca_full.explained_variance_ratio_, color='steelblue')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Individual Explained Variance')

axes[1].plot(range(1, len(cum_var)+1), cum_var, 'bo-')
axes[1].axhline(0.85, color='red', ls='--', label='85% threshold')
axes[1].axvline(n_85, color='orange', ls='--', label=f'{n_85} components')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Components for >=85% variance: {n_85}')

In [ ]:
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_true, cmap='RdYlGn_r', alpha=0.5, s=10)
plt.colorbar(sc, ax=axes[0], label='Machine Failure')
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% var)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% var)')
axes[0].set_title('PCA 2D — Colored by Failure')

colors = ['#3498db', '#2ecc71', '#e74c3c']
for t, (col, lbl) in enumerate(zip(colors, ['L', 'M', 'H'])):
    mask = machine_type == t
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=col, label=lbl, alpha=0.5, s=10)
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% var)')
axes[1].set_title('PCA 2D — Colored by Machine Type')
axes[1].legend(title='Type')

plt.tight_layout()
plt.show()

## 3. t-SNE — Non-linear Dimensionality Reduction

t-SNE preserves local neighborhood structure. Uses a 3000-sample subset for speed.


In [ ]:
np.random.seed(42)
idx = np.random.choice(len(X_scaled), size=min(3000, len(X_scaled)), replace=False)
X_sub = X_scaled[idx]
y_sub = y_true[idx]
type_sub = machine_type[idx]

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=500)
X_tsne = tsne.fit_transform(X_sub)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_sub, cmap='RdYlGn_r', alpha=0.6, s=15)
plt.colorbar(sc, ax=axes[0], label='Machine Failure')
axes[0].set_title('t-SNE — Colored by Failure')
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')

for t, (col, lbl) in enumerate(zip(colors, ['L', 'M', 'H'])):
    mask = type_sub == t
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=col, label=lbl, alpha=0.5, s=15)
axes[1].set_title('t-SNE — Colored by Machine Type')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].legend(title='Type')

plt.tight_layout()
plt.show()

## 4. K-Means Clustering

Elbow method + Silhouette score to find optimal k.


In [ ]:
k_range = range(2, 11)
inertias, silhouettes = [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels, sample_size=2000, random_state=42))

best_k = list(k_range)[int(np.argmax(silhouettes))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(k_range), inertias, 'bo-')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method')

axes[1].plot(list(k_range), silhouettes, 'go-')
axes[1].axvline(best_k, color='red', ls='--', label=f'Best k={best_k}')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs k')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f'Best k by Silhouette: {best_k}')

In [ ]:
km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
km_labels = km_best.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, km_labels, sample_size=2000, random_state=42)
db  = davies_bouldin_score(X_scaled, km_labels)
print(f'K-Means (k={best_k}): Silhouette={sil:.4f} | Davies-Bouldin={db:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=km_labels, cmap='tab10', alpha=0.5, s=10)
plt.colorbar(sc, ax=axes[0], label='Cluster')
axes[0].set_title(f'K-Means (k={best_k}) on PCA Space')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

unique, counts = np.unique(km_labels, return_counts=True)
axes[1].bar([f'Cluster {i}' for i in unique], counts,
            color=sns.color_palette('tab10', len(unique)))
axes[1].set_title('K-Means Cluster Sizes')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 5. DBSCAN Clustering


In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=10)
db_labels = dbscan.fit_predict(X_pca)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = int((db_labels == -1).sum())
print(f'DBSCAN: {n_clusters} clusters | {n_noise} noise points ({n_noise/len(db_labels)*100:.1f}%)')

plt.figure(figsize=(9, 6))
unique_db = np.unique(db_labels)
pal = plt.cm.tab10(np.linspace(0, 1, max(len(unique_db), 2)))
for uid, color in zip(unique_db, pal):
    mask = db_labels == uid
    lbl = 'Noise' if uid == -1 else f'Cluster {uid}'
    alpha = 0.3 if uid == -1 else 0.6
    s = 5 if uid == -1 else 15
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=[color], label=lbl, alpha=alpha, s=s)
plt.title('DBSCAN Clusters on PCA Space')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Hierarchical Clustering — Dendrogram


In [ ]:
sample_idx = np.random.choice(len(X_scaled), size=200, replace=False)
Z = linkage(X_scaled[sample_idx], method='ward')

plt.figure(figsize=(14, 5))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=90, leaf_font_size=8, show_contracted=True)
plt.title('Hierarchical Clustering Dendrogram (Ward, n=200 sample)')
plt.xlabel('Sample index')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

## 7. Comparative Analysis

### PCA vs t-SNE
| | PCA | t-SNE |
|-|-----|-------|
| **Type** | Linear | Non-linear |
| **Speed** | Fast | Slow |
| **Interpretable axes** | Yes (loadings) | No |
| **Best for** | Global structure, variance explained | Cluster visualization |

### K-Means vs DBSCAN
| | K-Means | DBSCAN |
|-|---------|--------|
| **Cluster shape** | Spherical | Arbitrary |
| **Noise handling** | None | Labels as noise |
| **Key hyperparameter** | k | eps, min_samples |
| **Scalability** | Good | Moderate |

### Real-world applications
- **Customer segmentation**: K-Means groups machines by operational profile
- **Anomaly detection**: DBSCAN identifies unusual sensor readings as noise
- **Dashboard visualization**: PCA 2D projection for operator monitoring

## 8. Ethical and Practical Considerations

| Risk | Description | Mitigation |
|------|-------------|------------|
| **Minority misrepresentation** | 3.4% failure cluster may merge into normal clusters | Validate cluster purity with failure labels post-hoc |
| **Dimensionality reduction hides structure** | PCA may compress failure-related variance | Compare explained variance vs domain knowledge |
| **DBSCAN sensitivity** | eps choice dramatically changes results | Silhouette-guided grid search |
| **Unsupervised != ground truth** | Clusters do not auto-map to meaningful categories | Validate against domain expert knowledge |
